# 01 · Do chlorophyllin and phthalocyanine mix linearly?

**Data.** Lin et al. (2022), *Sol. Energy Mater. Sol. Cells* 243, 111788 (Shi's ref. [31]), Fig. 2: absorption of chlorophyllin:phthalocyanine solutions in THF. This is the original source of Shi (2025) Fig. 9.

**Design of that experiment.** Chlorophyllin is held at 0.01 mg/mL in every sample. Phthalocyanine is *added* at 0, 0.005, 0.010, 0.020, 0.030 mg/mL (labeled 1:0 … 1:3). The pure phthalocyanine sample (0:1) has an unstated concentration.

**If the two do not interact** (Beer–Lambert additivity), every blend should equal

  A_blend = w_chl · A(1:0) + w_pc · A(0:1)

with **w_chl ≈ 1 in every blend** (same chlorophyllin) and **w_pc proportional to the phthalocyanine added** (0.5 : 1 : 2 : 3). Both checks work without knowing the 0:1 concentration.

**Status:** waiting for digitized curves in `data/raw/lin2022_ref31/`.

In [ ]:
REF31 = {
    "citation": "Lin, Wang, Lyu, Deng & Shi, Sol. Energy Mater. Sol. Cells 243, 111788 (2022)",
    "sample_form": "solution (Fig. 2); films are Fig. 5",
    "solvent": "THF",
    "chlorophyllin_mg_per_mL": 0.01,                       # fixed in every blend
    "phthalocyanine_mg_per_mL": {"1-0": 0.0, "1-0.5": 0.005, "1-1": 0.010, "1-2": 0.020, "1-3": 0.030},
    "pure_phthalocyanine_mg_per_mL": None,                  # 0:1 sample, not stated in the paper
    "chlorophyllin_species": "chlorophyllin sodium copper (Sigma-Aldrich)",
    "phthalocyanine_species": "beta-form copper phthalocyanine (TCI)",
    "y_axis": "Absorption intensity (a.u.)",
    "note": "Fig. 2 panel letters appear swapped vs. caption; Pc data = panel labeled (b)",
}

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from oshn_materials import spectra, mixing, metrics

WL = spectra.GRID

In [ ]:
RAW = spectra.DATA / "raw" / "lin2022_ref31"
# Hand-digitized files (fig2_*.csv in RAW) take priority; otherwise use the
# color-extracted files in RAW/extracted (see scripts/extract_lin2022_fig2b.py).
LABELS = ["1-0", "1-0.5", "1-1", "1-2", "1-3", "0-1"]
curves, source = {}, {}
for label in LABELS:
    for folder, tag in [(RAW, "hand"), (RAW / "extracted", "color-extracted")]:
        path = folder / f"fig2_{label}.csv"
        if path.exists():
            curves[label] = spectra.to_grid(*spectra.load_two_column(path))
            source[label] = tag
            break
print("Loaded:", source or "nothing yet")
print("Still needed:", [k for k in LABELS if k not in curves])

## Test 1 · Can any linear mix reproduce each blend?

In [ ]:
BLENDS = ["1-0.5", "1-1", "1-2", "1-3"]
rows = []
if {"1-0", "0-1"} <= curves.keys():
    a_chl, a_pc = curves["1-0"], curves["0-1"]
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
    for ax, label in zip(axes, BLENDS):
        if label not in curves:
            ax.set_title(f"{label}: not digitized yet"); continue
        w_chl, w_pc, err = mixing.fit_weights(curves[label], a_chl, a_pc)
        fit = mixing.linear_mix(a_chl, a_pc, w_chl, w_pc)
        rows.append({"blend": label,
                     "Pc_added_mg_per_mL": REF31["phthalocyanine_mg_per_mL"][label],
                     "w_chl": w_chl, "w_pc": w_pc, "RMSE": err,
                     "RMSE_over_peak": err / np.nanmax(curves[label])})
        ax.plot(WL, curves[label], color="#1d2320", label="measured (digitized)")
        ax.plot(WL, fit, "--", color="#3f6b4a", label="best linear fit")
        ax.plot(WL, curves[label] - fit, color="#b494e0", lw=1, label="residual")
        ax.set_title(label.replace("-", ":")); ax.set_xlabel("Wavelength (nm)")
    axes[0].set_ylabel("Absorption (a.u., as plotted)"); axes[0].legend(fontsize=8)
    fig.tight_layout(); fig.savefig("../figures/01_linear_fit.png", dpi=150)
    results = pd.DataFrame(rows)
    display(results.round(4))
else:
    print("Needs the pure 1:0 and 0:1 curves first.")

**Reading it.** A flat residual near zero supports linear mixing. Structure in the residual (a shifted peak, a new band) points to interaction. The paper reports the 404 nm peak shifting toward 380 nm as phthalocyanine is added; overlapping bands *can* shift an apparent peak without any interaction, and this test distinguishes the two.

## Test 2 · Do the weights follow the recipe?

In [ ]:
if rows:
    added = results["Pc_added_mg_per_mL"].to_numpy()
    w_pc = results["w_pc"].to_numpy()
    slope = np.sum(added * w_pc) / np.sum(added ** 2)   # best line through the origin
    results["w_pc_expected"] = slope * added
    print(f"w_chl across blends: {results['w_chl'].round(3).tolist()}  (linear mixing predicts ~1 each)")
    display(results[["blend", "Pc_added_mg_per_mL", "w_pc", "w_pc_expected"]].round(4))
    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    ax.plot(added, w_pc, "o", color="#1d2320", label="fitted")
    xs = np.linspace(0, added.max(), 50); ax.plot(xs, slope * xs, "--", color="#3f6b4a", label="proportional")
    ax.set_xlabel("Phthalocyanine added (mg/mL)"); ax.set_ylabel("w_pc"); ax.legend()
    fig.tight_layout(); fig.savefig("../figures/01_proportionality.png", dpi=150)
else:
    print("Runs after Test 1 has data.")

## Limits
- Digitized values carry reading error, especially where curves overlap.
- β-copper phthalocyanine dissolves poorly in THF. If some of it did not dissolve, the pure 0:1 spectrum could understate it, which would look like "non-linearity" but is really a solubility effect.
- These are solution spectra. The films in Fig. 5 were each adjusted to 85% AVT, so they are not on a same-amount basis.
- Lin et al. study photothermal heating, not photovoltaics; nothing here estimates electrical output.